# Notebook 01 — Protein Structure Visualization

**Learning objectives:**
- Load a protein structure from the PDB using Python
- Visualize it in 3D using py3Dmol inside this notebook
- Color by chain, secondary structure, and pLDDT
- Highlight specific residues (the PD-L1 hotspots)
- Compare two different structures side by side

**Time:** ~45 minutes

**Companion wiki page:** [3.1 Tour of Mol*](https://rucha1796.github.io/internship-bindcraft-2026/m3_01_molstar_tour/) and [3.3 PD-L1 Interface](https://rucha1796.github.io/internship-bindcraft-2026/m3_03_pdl1_interface/)

---
> **Before you start:** Make sure you have a GPU runtime. Go to **Runtime → Change runtime type → T4 GPU**. This notebook doesn't need a GPU, but it's good practice.

## Cell 1 — Install py3Dmol

py3Dmol is the library that lets us show 3D protein structures right inside this notebook.

In [ ]:
!pip install py3Dmol requests -q
import py3Dmol
import requests
print("✓ py3Dmol installed")

## Cell 2 — Download PDB structures

We'll download three PD-L1 structures directly from the RCSB PDB:
- **5C3T** — PD-L1 alone (high resolution, good reference)
- **5X8M** — PD-L1 in complex with PD-1 (natural interface)
- **8AOK** — PD-L1 with a designed miniprotein binder (your benchmark)

In [ ]:
def download_pdb(pdb_id):
    """Download a PDB structure as a string."""
    url = f"https://files.rcsb.org/download/{pdb_id.upper()}.pdb"
    response = requests.get(url)
    if response.status_code == 200:
        print(f"✓ Downloaded {pdb_id.upper()}")
        return response.text
    else:
        print(f"✗ Failed to download {pdb_id} (status {response.status_code})")
        return None

pdb_5c3t = download_pdb("5c3t")  # PD-L1 alone
pdb_5x8m = download_pdb("5x8m")  # PD-L1 + PD-1 complex
pdb_8aok = download_pdb("8aok")  # PD-L1 + designed binder

## Cell 3 — Count atoms and chains

Before visualizing, let's look at what's in each structure.

In [ ]:
def summarize_pdb(pdb_string, name):
    """Print a quick summary of a PDB file."""
    lines = pdb_string.split("\n")
    atom_lines = [l for l in lines if l.startswith("ATOM")]
    chains = sorted(set(l[21] for l in atom_lines if len(l) > 21))
    residues = sorted(set((l[21], l[22:26].strip()) for l in atom_lines if len(l) > 26))
    
    # Count unique residues per chain
    chain_residue_count = {}
    for chain, resnum in residues:
        chain_residue_count[chain] = chain_residue_count.get(chain, 0) + 1
    
    print(f"=== {name} ===")
    print(f"  Total ATOM records: {len(atom_lines)}")
    print(f"  Chains: {chains}")
    for chain, count in chain_residue_count.items():
        print(f"    Chain {chain}: {count} residues")
    print()

summarize_pdb(pdb_5c3t, "5C3T (PD-L1 alone)")
summarize_pdb(pdb_5x8m, "5X8M (PD-L1 + PD-1)")
summarize_pdb(pdb_8aok, "8AOK (PD-L1 + designed binder)")

## Cell 4 — Visualize PD-L1 alone (5C3T)

Cartoon representation, colored by chain (only chain A here).

**Controls:**
- Left-click + drag = rotate
- Scroll = zoom
- Right-click + drag = translate

In [ ]:
view = py3Dmol.view(width=750, height=500)
view.addModel(pdb_5c3t, "pdb")

# Cartoon ribbon representation
view.setStyle({"cartoon": {"color": "#3c5b6f"}})

view.zoomTo()
view.setBackgroundColor("white")
view.show()

print("PD-L1 structure (5C3T) — can you see the beta-sandwich fold?")
print("Rotate the structure. Notice the flat sheets and connecting loops.")

## Cell 5 — Color by secondary structure

Now let's color helices, sheets, and loops differently so you can identify secondary structure elements.

In [ ]:
view2 = py3Dmol.view(width=750, height=500)
view2.addModel(pdb_5c3t, "pdb")

# Color by secondary structure
view2.setStyle({"cartoon": {"colorscheme": "ssJmol"}})
# ssJmol coloring: helices = magenta, sheets = yellow, loops = white/gray

view2.zoomTo()
view2.setBackgroundColor("white")
view2.show()

print("Color key (ssJmol): helices = magenta/pink, sheets = yellow, loops = white")
print("Question: Is PD-L1 mostly helical or mostly sheet?")

## Cell 6 — Highlight the hotspot residues

Now let's see where the 5 hotspot residues (54, 56, 111, 115, 123) are located on PD-L1.

They'll appear as **yellow spheres** against the blue cartoon.

In [ ]:
HOTSPOT_RESIDUES = [54, 56, 111, 115, 123]
HOTSPOT_CHAIN = "A"

view3 = py3Dmol.view(width=750, height=500)
view3.addModel(pdb_5c3t, "pdb")

# Draw the whole protein as a cartoon in muted blue
view3.setStyle({"cartoon": {"color": "#3c5b6f", "opacity": 0.8}})

# Highlight each hotspot residue as a yellow sphere
for resnum in HOTSPOT_RESIDUES:
    view3.addStyle(
        {"chain": HOTSPOT_CHAIN, "resi": str(resnum)},
        {"sphere": {"color": "yellow", "radius": 1.0}}
    )

view3.zoomTo()
view3.setBackgroundColor("white")
view3.show()

print(f"Yellow spheres = hotspot residues: {HOTSPOT_RESIDUES}")
print("Are they on the surface? Are they near each other?")

## Cell 7 — The PD-L1 / PD-1 complex (5X8M)

Now load the complex where PD-1 is bound to PD-L1. We'll color each chain differently:
- **Chain A** (PD-L1) = steel blue
- **Chain B** (PD-1) = coral red

Notice where the two proteins touch — that's the interface you want your binder to compete with.

In [ ]:
view4 = py3Dmol.view(width=750, height=500)
view4.addModel(pdb_5x8m, "pdb")

# PD-L1 (chain A) in steel blue
view4.setStyle({"chain": "A"}, {"cartoon": {"color": "#3c5b6f"}})

# PD-1 (chain B) in coral
view4.setStyle({"chain": "B"}, {"cartoon": {"color": "#c1440e"}})

# Hotspot residues as yellow spheres
for resnum in HOTSPOT_RESIDUES:
    view4.addStyle(
        {"chain": "A", "resi": str(resnum)},
        {"sphere": {"color": "yellow", "radius": 0.8}}
    )

view4.zoomTo()
view4.setBackgroundColor("white")
view4.show()

print("Blue = PD-L1 (chain A), Coral = PD-1 (chain B), Yellow = hotspot residues")
print("Are the yellow hotspot residues at the interface between the two proteins?")

## Cell 8 — The designed binder (8AOK)

Now load the structure of PD-L1 with a **designed miniprotein binder** — the same type of thing you'll create with BindCraft.

- **Chain A** (PD-L1) = steel blue  
- **Chain B** (designed binder) = rose gold

Notice how much smaller the binder is compared to PD-1!

In [ ]:
view5 = py3Dmol.view(width=750, height=500)
view5.addModel(pdb_8aok, "pdb")

# PD-L1 (chain A) in steel blue
view5.setStyle({"chain": "A"}, {"cartoon": {"color": "#3c5b6f"}})

# Designed binder (chain B) in rose gold — the color scheme used throughout this internship
view5.setStyle({"chain": "B"}, {"cartoon": {"color": "#B76E79"}})

# Hotspot residues as yellow spheres
for resnum in HOTSPOT_RESIDUES:
    view5.addStyle(
        {"chain": "A", "resi": str(resnum)},
        {"sphere": {"color": "yellow", "radius": 0.8}}
    )

view5.zoomTo()
view5.setBackgroundColor("white")
view5.show()

print("Blue = PD-L1, Rose = designed binder (8AOK), Yellow = hotspot residues")
print("Your BindCraft designs should look similar to this rose-colored binder.")

## Cell 9 — Add a transparent surface

Let's add a semi-transparent molecular surface to PD-L1 so you can see how the binder sits on it.

In [ ]:
view6 = py3Dmol.view(width=750, height=500)
view6.addModel(pdb_8aok, "pdb")

# PD-L1 cartoon
view6.setStyle({"chain": "A"}, {"cartoon": {"color": "#3c5b6f"}})

# Binder cartoon
view6.setStyle({"chain": "B"}, {"cartoon": {"color": "#B76E79"}})

# Add semi-transparent surface to PD-L1
view6.addSurface(
    py3Dmol.VDW,
    {"opacity": 0.4, "color": "#3c5b6f"},
    {"chain": "A"}
)

# Hotspot spheres
for resnum in HOTSPOT_RESIDUES:
    view6.addStyle(
        {"chain": "A", "resi": str(resnum)},
        {"sphere": {"color": "yellow", "radius": 0.8}}
    )

view6.zoomTo()
view6.setBackgroundColor("white")
view6.show()

print("The transparent surface shows the PD-L1 molecular surface.")
print("Does the binder sit in a groove, or on a flat face?")

## Cell 10 — Your observations

Answer these questions based on what you've seen. Edit this cell and fill in your answers.

**Q1:** Is PD-L1 primarily helical, sheet-like, or mixed? What did you see in the secondary structure coloring?

_Your answer:_ 

**Q2:** Where are the 5 hotspot residues (yellow spheres) relative to the overall PD-L1 structure? On the surface? In the core? In a loop?

_Your answer:_

**Q3:** How does the designed binder (8AOK, rose) compare to PD-1 (5X8M, coral) in terms of size and shape?

_Your answer:_

**Q4:** Do the yellow hotspot residues appear to be at the interface where the binder touches PD-L1?

_Your answer:_

**Q5:** Looking at the surface view — does the binder sit in a defined groove, or on a relatively flat surface?

_Your answer:_

## Bonus — Extract chain A only from 5C3T

This shows you how to filter a PDB file in Python — useful for preparing target PDB files for BindCraft.

In [ ]:
def extract_chain(pdb_string, chain_id):
    """Extract only a specific chain from a PDB file."""
    lines = pdb_string.split("\n")
    chain_lines = []
    for line in lines:
        if line.startswith("ATOM") and len(line) > 21 and line[21] == chain_id:
            chain_lines.append(line)
    chain_lines.append("END")
    return "\n".join(chain_lines)

# Extract just PD-L1 (chain A) from the complex structure
pdl1_only = extract_chain(pdb_5c3t, "A")

# Save to a file
with open("PDL1_5C3T_chainA.pdb", "w") as f:
    f.write(pdl1_only)

atom_count = pdl1_only.count("\nATOM")
print(f"✓ Saved PDL1_5C3T_chainA.pdb")
print(f"  {atom_count} ATOM records (chain A only)")
print(f"\nFirst 3 lines:")
print("\n".join(pdl1_only.split("\n")[:3]))

---
## Summary

In this notebook you:
- Downloaded real protein structures from the PDB using Python
- Visualized them in 3D inside the notebook using py3Dmol
- Colored by chain and secondary structure
- Identified the 5 PD-L1 hotspot residues as yellow spheres
- Compared the natural PD-1 binder (5X8M) to a designed miniprotein binder (8AOK)
- Learned to extract a single chain from a PDB file

**Next:** Notebook 02 — Running ColabFold to predict PD-L1's structure from sequence